# 03 — 双变量探索性分析

> 🎯 **适用场景**: 探索特征之间的关系，发现"什么因素驱动了什么结果"
> ⏱️ **预计学习时长**: 50 分钟
> 📌 **核心问题**: 价格和运费有关系吗？评分和配送速度有关吗？不同品类的客单价差多少？

---

## 分析路线图

1. **数值 × 数值** → 散点图 / 相关性热力图
2. **数值 × 分类** → 箱线图 / 分组统计
3. **分类 × 分类** → 交叉表 / 热力图


In [ ]:
# ═══════════════════════════════════════════
# 环境准备
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

DATA_DIR = "../data"
df_orders  = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv",
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date'])
df_items   = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
df_products= pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
df_payments= pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
df_reviews = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")

# ── 合并不常用关联 ──
df_ord_item = df_orders.merge(df_items, on='order_id', how='inner')
df_ord_rev  = df_orders.merge(df_reviews, on='order_id', how='inner')

# 翻译类别
df_trans = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")
df_ord_item = df_ord_item.merge(df_products[['product_id','product_category_name']], on='product_id', how='left')
df_ord_item = df_ord_item.merge(df_trans, on='product_category_name', how='left')
df_ord_item['category_en'] = df_ord_item['product_category_name_english'].fillna(df_ord_item['product_category_name'])

# 订单级金额（聚合 payments）
order_pay = df_payments.groupby('order_id')['payment_value'].sum().reset_index()
df_ord_full = df_ord_item.merge(order_pay, on='order_id', how='left')

print("✅ 数据准备完成")


## 一、数值 × 数值 分析

### 1.1 相关性热力图

📌 选几个核心数值字段，看它们之间的相关性。

In [ ]:
# 📌 相关性热力图
corr_cols = ['price', 'freight_value', 'payment_value', 'product_photos_qty']
corr_data = df_ord_full[corr_cols].dropna()
corr_matrix = corr_data.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap='RdYlBu_r', center=0, square=True,
            linewidths=1, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('数值特征相关性矩阵', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# 📌 显著相关的关系
print("📌 显著相关性:")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            print(f"  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: r = {r:.3f}")


### 1.2 价格 vs 运费 → 散点图

📌 这是一个自然的商业问题：商品越贵，运费也越贵吗？

In [ ]:
# 📌 价格 vs 运费 散点图 + 回归线
fig, ax = plt.subplots(figsize=(10, 8))

# 采样提高渲染速度 (⚡ 万级样本约0.5秒)
sample = df_ord_item.sample(min(10000, len(df_ord_item)), random_state=42)

sns.regplot(data=sample, x='price', y='freight_value',
            scatter_kws={'alpha': 0.3, 's': 10},
            line_kws={'color': '#d62728', 'linewidth': 2},
            ax=ax)

ax.set_xlim(0, sample['price'].quantile(0.99))
ax.set_ylim(0, sample['freight_value'].quantile(0.99))
ax.set_xlabel('Price (R$)', fontsize=12)
ax.set_ylabel('Freight Value (R$)', fontsize=12)
ax.set_title('价格 vs 运费 (带回归线)', fontsize=14, fontweight='bold')

# 计算相关系数
r, p = stats.pearsonr(df_ord_item['price'], df_ord_item['freight_value'])
ax.text(0.95, 0.05, f'Pearson r = {r:.3f}\np-value < 0.001',
        transform=ax.transAxes, ha='right', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

# 💭 反思：相关性只有 r≈0.18，说明什么？运费更可能由什么因素决定？


## 二、数值 × 分类 分析

### 2.1 各品类的价格对比

📌 不同品类价格差异多大？哪些品类是高客单价的？

In [ ]:
# 📌 各品类价格箱线图（取Top 10销量品类）
top10_cat = df_ord_item['category_en'].value_counts().head(10).index
data_top10 = df_ord_item[df_ord_item['category_en'].isin(top10_cat)]

# 按价格中位数排序
order_cat = data_top10.groupby('category_en')['price'].median().sort_values().index

fig, ax = plt.subplots(figsize=(14, 8))
sns.boxplot(data=data_top10, x='price', y='category_en',
            order=order_cat, palette='viridis', ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_xlim(0, data_top10['price'].quantile(0.95))
ax.set_xlabel('Price (R$)', fontsize=12)
ax.set_title('Top 10 品类价格分布对比', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/03_category_price_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

# 📌 每个品类的价格统计
print("📌 各品类价格中位数 (Top 10):")
for cat in order_cat:
    med = df_ord_item[df_ord_item['category_en']==cat]['price'].median()
    mean = df_ord_item[df_ord_item['category_en']==cat]['price'].mean()
    print(f"  {cat:30s}  median=R$ {med:7.1f}  mean=R$ {mean:7.1f}")


### 2.2 支付方式 vs 客单价

📌 不同支付方式的用户，消费金额有差异吗？

In [ ]:
# 📌 支付方式 vs 客单价
order_pay_full = pd.merge(df_payments, df_orders[['order_id']], on='order_id')

# 按订单聚合
order_total = order_pay_full.groupby(['order_id','payment_type'])['payment_value'].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=order_total, x='payment_type', y='payment_value',
            palette='Set2', ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_ylim(0, order_total['payment_value'].quantile(0.95))
ax.set_xlabel('')
ax.set_ylabel('订单金额 (R$)')
ax.set_title('支付方式 vs 客单价', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 📌 每种支付方式的平均客单价
print("📌 各支付方式平均客单价:")
print(order_total.groupby('payment_type')['payment_value'].mean().sort_values(ascending=False).to_string())


### 2.3 评分 vs 配送时长

📌 这是体验优化中最重要的关系之一：配送越慢，评分越低？

In [ ]:
# 📌 计算配送时长（天）
df_ord_rev['delivery_days'] = (df_ord_rev['order_delivered_customer_date'] -
                                df_ord_rev['order_purchase_timestamp']).dt.days

# 剔除异常值（>60天的极端情况）
valid = df_ord_rev[(df_ord_rev['delivery_days']>=0) & (df_ord_rev['delivery_days']<=60)]

# 📌 评分 vs 配送时长 散点图 + 箱线图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 散点图
sample = valid.sample(min(5000, len(valid)), random_state=42)
sns.regplot(data=sample, x='delivery_days', y='review_score',
            scatter_kws={'alpha': 0.2, 's': 8},
            line_kws={'color': '#d62728', 'linewidth': 2}, ax=axes[0])
axes[0].set_xlabel('配送时长 (天)')
axes[0].set_ylabel('评分 (1-5)')
axes[0].set_title('配送时长 vs 评分', fontsize=13, fontweight='bold')

# 箱线图（分组）
valid['delivery_group'] = pd.cut(valid['delivery_days'], bins=[0,3,7,14,30,60],
                                  labels=['0-3天','4-7天','8-14天','15-30天','30+天'])
sns.boxplot(data=valid, x='delivery_group', y='review_score',
            palette='RdYlGn', ax=axes[1])
axes[1].set_xlabel('配送时长分组')
axes[1].set_ylabel('评分')
axes[1].set_title('配送时长分组 vs 评分分布', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/03_delivery_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()

# 📌 每个配送组的平均评分
print("📌 配送时长 vs 平均评分:")
print(valid.groupby('delivery_group', observed=True)['review_score'].agg(['mean','count']))

r, p = stats.pearsonr(valid['delivery_days'], valid['review_score'])
print(f"\n皮尔逊相关系数: r = {r:.3f} (p < 0.001)")

# 💭 反思：配送时长对评分影响有多大？改善物流能提升多少用户体验？


## 📝 康奈尔笔记联动区

### 左侧栏（核心概念）
- **Pearson 相关系数**: 衡量线性关系强度（-1 ~ 1）
- **箱线图分组对比**: 快速发现不同分类下的数值差异
- **配送时长**: 用户体验的核心影响因素

### 右侧栏（反思问题）
> 💭 价格和运费相关性弱，运费更可能由重量/距离决定——我们有这些数据吗？
> 💭 boleto 用户的客单价反而更高？是巧合还是有业务解释（如批发商偏好）？
> 💭 配送超过 7 天后评分明显下降——Olist 的 SLA 应该是多少？

### 底部栏（行动清单）
- [ ] 验证 weight/distance 是否能解释运费（需要 geolocation 表）
- [ ] 对比 boleto vs credit_card 用户的画像差异
- [ ] 到 project01_reflections.md 记录双变量核心发现
